#### Logistic Regression & Random Forest

In [2]:
import pandas as pd

df = pd.read_csv("../../data/practice_data.csv")
print(df.shape)
df.head()   


(20, 4)


,url,label,len,dash
0,google.com,0,1,0
1,facebook.com,1,1,0
2,amazon.in,0,0,0
3,microsoft.com,0,1,0
4,secure-login-wellsfargo.xyz,1,0,2


In [3]:
df.isnull().sum()
df["label"].value_counts()
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   url     20 non-null     str  
 1   label   20 non-null     int64
 2   len     20 non-null     int64
 3   dash    20 non-null     int64
dtypes: int64(3), str(1)
memory usage: 772.0 bytes
None


In [4]:
X = df.drop(["label","url"], axis=1)
y = df["label"].replace(-1, 0)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


In [6]:
import joblib
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
joblib.dump(lr,'logistic.pkl')


['logistic.pkl']

In [7]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = lr.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         3
           1       1.00      1.00      1.00         3

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6



In [8]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
joblib.dump(rf,'random_forest_v1.pkl')
print("RF Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))



RF Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         3
           1       1.00      1.00      1.00         3

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6



In [9]:
import pandas as pd

importance = pd.Series(rf.feature_importances_, index=X.columns)
print(importance.sort_values(ascending=False))


dash    0.752957
len     0.247043
dtype: float64


In [10]:
print(lr.n_iter_)


[8]


In [11]:
import pandas as pd

# 1. The raw input
new_site = "helo-login.com"

# 2. Create the DataFrame and extract features IMMEDIATELY
new_df = pd.DataFrame([new_site], columns=['url'])

# Logic: You must build the 'len' and 'dash' columns for this new data!
new_df['len'] = new_df['url'].apply(len)
new_df['dash'] = new_df['url'].apply(lambda x: x.count('-'))

# 3. NOW you can filter to match the training columns
# This ensures 'url' is dropped and the order matches X_train exactly
new_df = new_df[X.columns]

# 4. Predict
prediction = rf.predict(new_df)

if prediction[0] == 1:
    print("Warning: This site looks like Phishing!")
else:
    print("This site looks Safe.")